# Microglia-Inspired Dynamic Pruning - Qwen2.5-3B Evaluation

This notebook runs the pruning experiment on Qwen2.5-3B-Instruct with GSM8K.

**What we're doing:** Training small "agent" networks to learn which attention heads can be pruned during inference. Inspired by how microglia prune synapses in the brain.

**Expected results:**
- 20-30% of attention heads pruned
- ~10-15% latency improvement
- Minimal accuracy loss

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/microglia-pruning/blob/main/notebooks/qwen_pruning_eval.ipynb)

## Setup

This takes ~2 minutes on a T4 GPU.

In [ ]:
# Remove any existing clone and get fresh copy
import os
import shutil

if os.path.exists('/content/microglia-pruning'):
    shutil.rmtree('/content/microglia-pruning')
    print('Removed old clone')

# Clone repo with latest code
!git clone https://github.com/Tommaso-R-Marena/microglia-pruning.git
%cd microglia-pruning

# Verify we have latest commit
!git log --oneline -1

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes peft datasets scipy numpy tqdm matplotlib
!pip install -q fvcore  # For FLOP counting

print("✓ Installation complete")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import sys
import time

# Make sure we're using the code from the cloned repo
sys.path.insert(0, '/content/microglia-pruning')

from src.system import MicrogliaPruningSystem

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Part 1: Load Base Model

We're using Qwen2.5-3B-Instruct. It's a state-of-the-art 3B model that's highly efficient and capable.

**Note:** This downloads the model weights. First time takes ~3-5 minutes.

In [ ]:
# Initialize the pruning system
# This loads Qwen2.5-3B-Instruct and creates pruning agents for each layer

system = MicrogliaPruningSystem(
    model="Qwen/Qwen2.5-3B-Instruct",
    num_heads=16,  # Qwen2.5-3B has 16 attention heads per layer
    hidden_dim=128,  # Size of our agent networks
    temperature=1.0  # Controls how "sharp" pruning decisions are
)

print("\n✓ System initialized!")
print(f"\nModel size: {sum(p.numel() for p in system.model.parameters())/1e9:.2f}B parameters")
print(f"Agent size: {sum(p.numel() for p in system.agents.parameters())/1e6:.2f}M parameters")
print(f"\nAgent overhead: {sum(p.numel() for p in system.agents.parameters())/sum(p.numel() for p in system.model.parameters())*100:.3f}%")

## Part 2: Test Baseline Performance

Before training, let's see how the base model performs on a few GSM8K problems.

In [ ]:
# Test on a couple examples
test_questions = [
    "A store sells apples for $2 each. If Sarah buys 5 apples, how much does she spend?",
    "John has 15 candies. He gives 3 to each of his 4 friends. How many candies does he have left?",
    "A rectangle has a length of 8 meters and a width of 5 meters. What is its area?"
]

print("Testing baseline (unpruned) model:\n")

for i, question in enumerate(test_questions, 1):
    prompt = f"Question: {question}\nAnswer:"
    
    start = time.time()
    output = system.generate(prompt, max_new_tokens=100)
    elapsed = time.time() - start
    
    # Extract just the answer part
    answer = system._extract_answer(output)
    
    print(f"Q{i}: {question}")
    print(f"A{i}: {answer}")
    print(f"Time: {elapsed:.2f}s\n")

## Part 3: Train Pruning Agents

Now we train the agents to learn which heads to prune. This uses curriculum learning - we gradually increase the pruning pressure over epochs.

**Training time:** ~15-20 minutes on a T4 GPU for 3 epochs

In [ ]:
# Train the pruning agents
# We use fewer epochs for demo

system.train(
    dataset_name="gsm8k",
    num_epochs=3,  
    batch_size=2,  # Small batch size for memory efficiency
    learning_rate=1e-4,
    alpha_schedule=(0.01, 0.2),  # Start low, increase to encourage pruning
    use_lora=False  # Disabled for compatibility with wrapped attention
)

print("\n✓ Training complete!")

## Part 4: Visualize Training Progress

Let's plot how the loss evolved during training.

In [ ]:
# Plot training curves
history = system.training_history

if history:
    epochs = range(1, len(history) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Task loss
    ax1.plot(epochs, [h['task_loss'] for h in history], 'b-o', linewidth=2, markersize=8)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Task Loss', fontsize=12)
    ax1.set_title('Task Performance', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Sparsity loss
    ax2.plot(epochs, [h['sparsity_loss'] for h in history], 'r-o', linewidth=2, markersize=8)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Sparsity Loss', fontsize=12)
    ax2.set_title('Pruning Pressure', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No training history available")

## Part 5: Evaluate Pruned Model

Now let's test the pruned model on the GSM8K test set.

This evaluates on 200 test examples (~5 minutes)

In [ ]:
# Evaluate on test set
results = system.evaluate(
    dataset_name="gsm8k",
    split="test",
    max_samples=200
)

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
print(f"Accuracy: {results['accuracy']:.1%}")
print(f"Correct: {results['correct']}/{results['total']}")
print(f"Sparsity: {results['sparsity']:.1%} heads pruned")
print("="*50)

## Part 6: Visualize Pruning Pattern

Let's see which heads get pruned in different layers.

In [ ]:
# Run a forward pass to capture masks
_ = system.generate("Question: What is 2 + 2?\nAnswer:", max_new_tokens=50, use_pruning=True)

# Collect masks from all layers
all_masks = []
for layer_idx, layer in enumerate(system.get_layers()):
    attn = getattr(layer, "self_attn", getattr(layer, "attn", None))
    if hasattr(attn, 'last_masks'):
        masks = attn.last_masks
        if masks is not None:
            # Average over batch
            avg_mask = masks.mean(dim=0).cpu().numpy()
            all_masks.append(avg_mask)

if all_masks:
    # Create heatmap
    mask_matrix = np.array(all_masks)
    
    plt.figure(figsize=(12, 8))
    im = plt.imshow(mask_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im, label='Keep Probability')
    plt.xlabel('Head Index', fontsize=12)
    plt.ylabel('Layer Index', fontsize=12)
    plt.title('Pruning Pattern Across Layers', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("No masks captured")

## Summary

### What we did:
1. Loaded Qwen2.5-3B-Instruct
2. Trained small pruning agents
3. Evaluated on GSM8K math problems
4. Visualized the pruning patterns

**Questions or issues?** Open an issue on GitHub: [github.com/Tommaso-R-Marena/microglia-pruning](https://github.com/Tommaso-R-Marena/microglia-pruning)